## Utils

In [ ]:
import asyncio
from typing import Any, Dict, List

from langsmith import Client
from langsmith.evaluation import evaluate
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

from run_pipeline import StaticParams
from src.all_functionality import load_eval_tasks
from src.evaluator import Evaluator

# Global setup
EXPERIMENT_PREFIX = "Offline-Notion-Eval"
DEFAULT_DATASET_NAME = "Eval_Run_Batch_1"
JUDGE_MODEL_NAME = "gemma27"

client = Client()
core_evaluator = Evaluator(default_judge_model=JUDGE_MODEL_NAME)
eval_tasks = load_eval_tasks("evals")

# Task -> evaluation spec dictionary assembled from eval YAML files
TASK_EVAL_SPECS: Dict[str, Dict[str, Any]] = {
    task_id: {
        "task": data.get("task", ""),
        "properties": data.get("properties", {}),
        "rag_eval_statements": data.get("rag_eval_statements", []),
        # Optional, currently not present in your YAMLs but supported for future use
        "code_eval_statements": data.get("rag_eval_statements", []),
    }
    for task_id, data in eval_tasks.items()
}


def _example_to_dict(example: Any) -> Dict[str, Any]:
    if isinstance(example, dict):
        return example
    out: Dict[str, Any] = {}
    out["inputs"] = getattr(example, "inputs", {}) or {}
    out["outputs"] = getattr(example, "outputs", {}) or {}
    return out


class RagStatementsEvaluator:
    """LangSmith-compatible evaluator for RAG statement verification only."""

    def __init__(self, evaluator: Evaluator, task_specs: Dict[str, Dict[str, Any]], judge_model_name: str | None = None):
        self.evaluator = evaluator
        self.task_specs = task_specs
        self.judge_model_name = judge_model_name

    async def _compute(self, example_like: Any) -> Dict[str, Any]:
        ex = _example_to_dict(example_like)
        outputs = ex.get("outputs", {})
        state = outputs.get("pre_computed_state", {})
        task_id = outputs.get("task_id") or ex.get("inputs", {}).get("task_id", "")

        retrieval_context = state.get("retrieval_context", "")
        statements = self.task_specs.get(task_id, {}).get("rag_eval_statements", [])

        result = await self.evaluator.eval_context_statements(
            context=retrieval_context,
            statements=statements,
            judge_model_name=self.judge_model_name,
        )

        status_items = result.get("statements", [])
        present_count = sum(1 for item in status_items if str(item.get("status", "")).strip().lower() == "present")
        score = (present_count / len(status_items)) if status_items else 0.0

        return {
            "key": "rag_statements_score",
            "score": score,
            "comment": f"task={task_id} present={present_count}/{len(status_items)}",
            "metadata": {"raw": result},
        }

    def __call__(self, *args: Any, **kwargs: Any) -> Dict[str, Any]:
        # Supports common LangSmith evaluator calling patterns.
        example_like = kwargs.get("example")
        if example_like is None and args:
            example_like = args[-1]
        return asyncio.run(self._compute(example_like))


class CodegenEvaluator:
    """LangSmith-compatible evaluator for code generation (execution + code statements)."""

    def __init__(self, evaluator: Evaluator, task_specs: Dict[str, Dict[str, Any]], judge_model_name: str | None = None):
        self.evaluator = evaluator
        self.task_specs = task_specs
        self.judge_model_name = judge_model_name

    async def _compute(self, example_like: Any) -> Dict[str, Any]:
        ex = _example_to_dict(example_like)
        outputs = ex.get("outputs", {})
        state = outputs.get("pre_computed_state", {})
        task_id = outputs.get("task_id") or ex.get("inputs", {}).get("task_id", "")

        code = state.get("final_code") or state.get("generated_code") or ""
        code_eval_statements = self.task_specs.get(task_id, {}).get("code_eval_statements", [])

        result = await self.evaluator.eval_code(
            code=code,
            statements=code_eval_statements,
            judge_model_name=self.judge_model_name,
        )

        execution_pass = bool(result.get("execution", {}).get("pass", False))
        status_items = result.get("statements", {}).get("statements", [])
        present_count = sum(1 for item in status_items if str(item.get("status", "")).strip().lower() == "present")
        statements_score = (present_count / len(status_items)) if status_items else 0.0

        score = 0.5 * (1.0 if execution_pass else 0.0) + 0.5 * statements_score

        return {
            "key": "codegen_score",
            "score": score,
            "comment": f"task={task_id} exec={execution_pass} statements={present_count}/{len(status_items)}",
            "metadata": {"raw": result},
        }

    def __call__(self, *args: Any, **kwargs: Any) -> Dict[str, Any]:
        # Supports common LangSmith evaluator calling patterns.
        example_like = kwargs.get("example")
        if example_like is None and args:
            example_like = args[-1]
        return asyncio.run(self._compute(example_like))


# Dummy target: we evaluate pre-computed states only
def dummy_target(inputs: Dict[str, Any]) -> Dict[str, Any]:
    return {"status": "precomputed", "task_id": inputs.get("task_id", "")}


rag_eval = RagStatementsEvaluator(core_evaluator, TASK_EVAL_SPECS, judge_model_name=JUDGE_MODEL_NAME)
codegen_eval = CodegenEvaluator(core_evaluator, TASK_EVAL_SPECS, judge_model_name=JUDGE_MODEL_NAME)

## Samples

In [ ]:
# Flexible sample loading + dataset creation in a single cell
# Change these variables directly for each run
DATASET_NAME = "Eval_Run_Batch_1"
THREAD_PREFIX_FILTERS = ["add_task", "retrieve_tasks"]  # [] means no filtering
MAX_EXAMPLES = 8

static_params = StaticParams()

states_to_evaluate: List[Dict[str, Any]] = []

async with AsyncSqliteSaver.from_conn_string(static_params.sqlite_saver_path) as checkpointer:
    # Gather unique thread IDs from checkpoint store
    thread_ids: List[str] = []
    seen_thread_ids = set()

    async for checkpoint in checkpointer.alist(None):
        thread_id = checkpoint.config.get("configurable", {}).get("thread_id")
        if not thread_id or thread_id in seen_thread_ids:
            continue

        if THREAD_PREFIX_FILTERS and not any(thread_id.startswith(prefix + "_") for prefix in THREAD_PREFIX_FILTERS):
            continue

        seen_thread_ids.add(thread_id)
        thread_ids.append(thread_id)

    # Most recent are generally appended later; keep this direct and easy to tweak
    selected_thread_ids = thread_ids[-MAX_EXAMPLES:]

    for thread_id in selected_thread_ids:
        config = {"configurable": {"thread_id": thread_id}}
        checkpoint_tuple = await checkpointer.aget_tuple(config)
        if not checkpoint_tuple:
            continue

        channel_values = checkpoint_tuple.checkpoint.get("channel_values", {})
        task_id = thread_id.rsplit("_", 1)[0] if "_" in thread_id else thread_id

        states_to_evaluate.append({
            "thread_id": thread_id,
            "task_id": task_id,
            "query": channel_values.get("user_prompt", ""),
            "pre_computed_state": channel_values,
        })

# Create or fetch dataset
existing = None
for ds in client.list_datasets(dataset_name=DATASET_NAME):
    if ds.name == DATASET_NAME:
        existing = ds
        break

dataset = existing or client.create_dataset(dataset_name=DATASET_NAME)

# Push examples
for state in states_to_evaluate:
    client.create_example(
        inputs={
            "query": state["query"],
            "task_id": state["task_id"],
            "thread_id": state["thread_id"],
        },
        outputs={
            "task_id": state["task_id"],
            "pre_computed_state": state["pre_computed_state"],
        },
        dataset_id=dataset.id,
    )

print(f"Dataset: {dataset.name} | uploaded examples: {len(states_to_evaluate)}")
states_to_evaluate[:2]

## Evaluate

In [ ]:
results = evaluate(
    dummy_target,
    data=DATASET_NAME,
    evaluators=[rag_eval, codegen_eval],
    experiment_prefix=EXPERIMENT_PREFIX,
)
results